In [116]:
# Code to map Maven elements to Origami's InterpretiveLab elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# May 2026

In [117]:
# For each Maven data element, check if there is an exact or close matching data element in the origami InterpretiveLab package
# The output will list the maven element alongside the origami element and other supporting attributes

In [118]:
from sentence_transformers import SentenceTransformer


In [119]:
import pandas as pd
import numpy as np

In [120]:
# Read maven elements
df_maven = pd.read_csv('output/all_db_elements_use_this.csv',na_filter=False)
len(df_maven)

6232

In [121]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   QUESTION_ID     6232 non-null   object
 1   value           6232 non-null   object
 2   QUESTIONSET_ID  6232 non-null   object
 3   MODEL_NAME      6232 non-null   object
dtypes: object(4)
memory usage: 194.9+ KB


In [122]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [123]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [124]:
# Clean up questions or data elements
df_maven.loc[:,'QUESTION_ID'] =[ clean_up(q) for q in df_maven.loc[:,'QUESTION_ID']]

In [125]:
df_maven.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1361
DiseaseSurveillanceModel_Hep             754
Childhood_Lead_Child_Model               386
ClusterModel                             205
Childhood_Lead_Investigation_Model       136
DiseaseSurveillanceModel_Coinfection     114
PortalApplicationModel                    41
EHRAndScreeningModel_STD                  22
DiseaseSurveillanceModel_AggNETSS         14
IsolationAndControlModel                   7
Name: count, dtype: int64

In [126]:
# General clean up
df_maven.loc[:,'MODEL_NAME'] = [m.strip() for m in df_maven.loc[:,'MODEL_NAME']]

In [127]:
# creating a sorting order for model name which is then used to sort the dataframe 
# Using the order from value counts above
sorter = ['DiseaseSurveillanceModel_General','DiseaseSurveillanceModel_STD','DiseaseSurveillanceModel_Hep','Childhood_Lead_Child_Model',
          'ClusterModel','Childhood_Lead_Investigation_Model','DiseaseSurveillanceModel_Coinfection','PortalApplicationModel',
          'EHRAndScreeningModel_STD','DiseaseSurveillanceModel_AggNETSS','IsolationAndControlModel']

In [128]:
# convert model_name column to category
df_maven.MODEL_NAME = df_maven.MODEL_NAME.astype('category')
# Set sorter as the categories hierarchy
df_maven.MODEL_NAME = df_maven.MODEL_NAME.cat.set_categories(sorter)


In [129]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   QUESTION_ID     6232 non-null   object  
 1   value           6232 non-null   object  
 2   QUESTIONSET_ID  6232 non-null   object  
 3   MODEL_NAME      6232 non-null   category
dtypes: category(1), object(3)
memory usage: 152.7+ KB


In [130]:
# Sort df on model_name
df_maven = df_maven.sort_values(['MODEL_NAME'])

In [131]:
df_maven = df_maven.reset_index(drop=True)

In [132]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
6227,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6229,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6230,REPORT_CATEGORY_PROVIDER,Urgent,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [133]:
# duplicates
df_maven[df_maven.duplicated(['QUESTION_ID'])]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
3207,RACE_NATIVE_HAWAIIAN_PACIFIC_ISLANDER_DETAIL,2500-7,DEMOGRAPHIC,DiseaseSurveillanceModel_STD
3218,CDR_CONTACT_TYPE_RELATION_OTHER_DESC,Boyfriend/Babies Daddy,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_STD
3249,CDR_DATA_SUBMITTED_DATE,12/28/2022,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_STD
3267,ADDRESS_TYPE,3,REPORTING_MORBIDITY,DiseaseSurveillanceModel_STD
3305,RACE_REFUSED,NO,DEMOGRAPHIC,DiseaseSurveillanceModel_STD
...,...,...,...,...
6222,EPISODE_DATE,12/31/2016,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6224,MMWR_WEEK,53,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6226,ISO_QUAR_CONTACT_INSTRUCTIONS,"Surveillance, testing, counseling, interview, ...",ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [134]:
# Keeping the first appearance of question
df_maven = df_maven.drop_duplicates(['QUESTION_ID'])

In [135]:
df_maven[df_maven.duplicated(['QUESTION_ID'])]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME


In [136]:
df_maven = df_maven.reset_index(drop=True)

In [137]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
5439,COMMENTS,whatever,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
5440,PRODUCT_CODE,YER,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5441,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5442,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [138]:
# HealthcareUtilization elements
df_il = pd.read_csv('origami/csv/Custom__InterpretiveLab.csv',na_filter=False)
len(df_il)

138

In [139]:
df_il.loc[:,'Label'].value_counts(dropna=False)

Label
                                          65
[LAB628]Test Result Quantitative           2
[LAB115]Result Units                       2
Interpretive Lab ID                        1
[82773-3]Time of Lab Result                1
                                          ..
[INV291]Test Result                        1
[41852-5]Organism                          1
[68963-8]Specimen Collection Date/Time     1
[85930-6]Date specimen sent to CDC         1
[31208-2_STD] Specimen Source              1
Name: count, Length: 72, dtype: int64

In [140]:
# Get the two columns of interest and non-blank values
df_il = df_il[['Field','Label']]
df_il = df_il[df_il['Label']!='']
len(df_il)

73

In [141]:
df_il = df_il.reset_index(drop=True)

In [142]:
df_il

,Field,Label
0,InterpretiveLabID,Interpretive Lab ID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
...,...,...
68,CustomCode31ID,[LAB278_STD] Organism Name
69,CustomCode32ID,[INV291_STD] Test Result
70,CustomCode33ID,[STD123_STD] Nontreponemal serologic syphilis ...
71,CustomCode34ID,[LAB115_STD] Units of Measure


In [143]:
# Original label to be used later for output
df_il.rename(columns={'Label':'original_label'},inplace=True)
df_il.columns

Index(['Field', 'original_label'], dtype='object')

In [144]:
# Remove only the first occurrence of [] in a string
def remove_squares(somestring):
    if ']' in somestring:
        for i in range(len(somestring)):
            if somestring[i]==']':
                return somestring[i+1:].strip()
    return somestring.strip()


In [145]:
remove_squares('asd[asdasd]asc[obas]da')

'asc[obas]da'

In [146]:
# Process the column values (eliminate the values in square brackets)
df_il.loc[:,'Label'] = [ remove_squares(label) for label in df_il.loc[:,'original_label']]

In [147]:
# Clean up column names in both data sets
df_il.loc[:,'Label'] =[ clean_up(q) for q in df_il.loc[:,'Label']]

In [148]:
df_il.loc[:,'Label'].value_counts()

Label
Test Type                                        4
Test Result Quantitative                         3
Test Result                                      3
Result Units                                     2
Specimen Collection Date                         2
Date of Lab Result                               2
Test Method                                      2
Specimen Type                                    2
Organism Name                                    2
Other result                                     1
Interpretive Lab ID                              1
Other test                                       1
Other specimen source                            1
Specimen Analyzed Time                           1
Specimen Received Time                           1
Specimen Collection Time                         1
Time of Lab Result                               1
Other performing lab                             1
Specimen Received Date                           1
Other Test Type          

In [149]:
# duplicates labels exist 
df_il[df_il.duplicated(['Label'])]

,Field,original_label,Label
25,CustomCode18ID,[LAB115]Result Units,Result Units
29,CustomCode22ID,[INV290]Test Type,Test Type
30,CustomCode23ID,[85069-3]Test Method,Test Method
31,CustomCode24ID,[INV291]Test Result,Test Result
41,CustomDate9,[82772-5_STD] Date of Lab Result,Date of Lab Result
42,CustomDate10,[33882-2_STD] Specimen Collection Date,Specimen Collection Date
49,CustomText3,[LAB628]Test Result Quantitative,Test Result Quantitative
62,CustomText16,[LAB628_STD] Test Result Quantitative,Test Result Quantitative
63,CustomCode26ID,[66746-9]Specimen Type,Specimen Type
66,CustomCode29ID,[INV290_Measles]Test Type,Test Type


In [150]:
# dropping them
df_il = df_il.drop_duplicates(['Label'])

In [151]:
df_il[df_il.duplicated(['Label'])]

,Field,original_label,Label


In [152]:
df_il = df_il.reset_index(drop=True)
df_il

,Field,original_label,Label
0,InterpretiveLabID,Interpretive Lab ID,Interpretive Lab ID
1,ClientID,Client,Client
2,EntryUserID,Entry User,Entry User
3,EntryDate,Entry Date,Entry Date
4,ModifiedUserID,Modified User,Modified User
5,ModifiedDate,Modified Date,Modified Date
6,IncidentTypeID,Incident Type,Incident Type
7,IncidentID,Incident,Incident
8,CustomCode1ID,[INV290]Type of test(s) performed for this cas...,Type of tests performed for this case V
9,CustomCode2ID,[INV291]Test Result (Varicella),Test Result Varicella


In [153]:
# Do a semantic match between Maven elements and Origami HAI labels

In [154]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven.QUESTION_ID]
origami_elements = [str.replace(col,'_',' ') for col in df_il.Label]

In [155]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [156]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [157]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

60
5444


array([[-0.16024935,  0.0271387 ,  0.06354488, ..., -0.00712903,
        -0.01673306,  0.06124594],
       [ 0.04245438,  0.03854417,  0.0684232 , ..., -0.06141916,
         0.08061311, -0.07027474],
       [-0.08168999,  0.03473966, -0.06668424, ..., -0.04572672,
        -0.00667135,  0.0398071 ],
       ...,
       [-0.06123391,  0.04124329, -0.09608547, ...,  0.11074385,
         0.03158536, -0.0520657 ],
       [ 0.03361172,  0.06674883, -0.07311527, ...,  0.01481686,
         0.04655252, -0.02631201],
       [-0.00503135,  0.02209355, -0.08806273, ..., -0.04621199,
         0.04993448,  0.04251292]], dtype=float32)

In [158]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [159]:
similarities

tensor([[ 0.0768,  0.1604,  0.0948,  ..., -0.0763,  0.0856,  0.0243],
        [ 0.0069,  0.0250,  0.0455,  ...,  0.0336,  0.0245,  0.0976],
        [ 0.2250,  0.2240,  0.2282,  ...,  0.0644,  0.0817,  0.0792],
        ...,
        [ 0.1609,  0.2358,  0.2094,  ..., -0.0443,  0.0508,  0.0598],
        [ 0.4553,  0.0612,  0.0157,  ...,  0.1583,  0.0885,  0.2108],
        [ 0.1857,  0.1426,  0.0463,  ...,  0.0579,  0.0432,  0.0056]])

In [160]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


5444

In [161]:
df_similarities = pd.DataFrame(similarities,index=df_maven.QUESTION_ID,columns=df_il.original_label)

In [162]:
df_similarities.to_csv('output/similarity_scores_InterpretiveLab.csv')

In [163]:
df_maven = df_maven.reset_index(drop=True)
df_maven.columns

Index(['QUESTION_ID', 'value', 'QUESTIONSET_ID', 'MODEL_NAME'], dtype='object')

In [164]:
df_output = df_maven[['MODEL_NAME','QUESTION_ID','QUESTIONSET_ID']]

In [165]:
#df_similarities.info()

In [166]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.75
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'QUESTION_ID':idx,'target_label':max_col})



In [167]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

5444


In [168]:
df_merge = pd.merge(df_filtered,df_il,left_on='target_label',right_on='original_label',how='left')

In [169]:
df_merge

,QUESTION_ID,target_label,Field,original_label,Label
0,CURRENT_TELEPHONE_WORK,,NaN,NaN,NaN
1,EAT_YOGURT_PURCHASED_FROM,,NaN,NaN,NaN
2,CDR_PERSON_OR_AGENCY_REPORTING,,NaN,NaN,NaN
3,ANTACID_EXPOSURE,,NaN,NaN,NaN
4,DIARY_DAY_1_DINNER,,NaN,NaN,NaN
...,...,...,...,...,...
5439,COMMENTS,,NaN,NaN,NaN
5440,PRODUCT_CODE,,NaN,NaN,NaN
5441,PROGRAM_CODE,,NaN,NaN,NaN
5442,REPORT_CATEGORY_LAB,,NaN,NaN,NaN


In [170]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='QUESTION_ID')
#df_final

In [171]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5444 entries, 0 to 5443
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   MODEL_NAME      5444 non-null   category
 1   QUESTION_ID     5444 non-null   object  
 2   QUESTIONSET_ID  5444 non-null   object  
 3   target_label    5444 non-null   object  
 4   Field           26 non-null     object  
 5   original_label  26 non-null     object  
 6   Label           26 non-null     object  
dtypes: category(1), object(6)
memory usage: 261.0+ KB


In [172]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Custom__InterpretiveLab' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [173]:
df_final = df_final[["MODEL_NAME",'QUESTION_ID','target_table','target_field','target_label']]
#df_final

In [174]:
df_final

,MODEL_NAME,QUESTION_ID,target_table,target_field,target_label
0,DiseaseSurveillanceModel_General,CURRENT_TELEPHONE_WORK,,NaN,
1,DiseaseSurveillanceModel_General,EAT_YOGURT_PURCHASED_FROM,,NaN,
2,DiseaseSurveillanceModel_General,CDR_PERSON_OR_AGENCY_REPORTING,,NaN,
3,DiseaseSurveillanceModel_General,ANTACID_EXPOSURE,,NaN,
4,DiseaseSurveillanceModel_General,DIARY_DAY_1_DINNER,,NaN,
...,...,...,...,...,...
5439,DiseaseSurveillanceModel_AggNETSS,COMMENTS,,NaN,
5440,IsolationAndControlModel,PRODUCT_CODE,,NaN,
5441,IsolationAndControlModel,PROGRAM_CODE,,NaN,
5442,IsolationAndControlModel,REPORT_CATEGORY_LAB,,NaN,


In [175]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1221
Childhood_Lead_Child_Model               361
DiseaseSurveillanceModel_Hep             296
ClusterModel                             195
Childhood_Lead_Investigation_Model       125
PortalApplicationModel                    30
DiseaseSurveillanceModel_Coinfection      10
EHRAndScreeningModel_STD                   8
IsolationAndControlModel                   4
DiseaseSurveillanceModel_AggNETSS          2
Name: count, dtype: int64

In [176]:
# Convert category type back to str object
df_final.MODEL_NAME = df_final.MODEL_NAME.astype('str')

In [177]:
# Rename model names
df_final.loc[:,'MODEL_NAME'] = [ m.split('_')[1] if 'DiseaseSurveillanceModel_' in m else m for m in df_final.loc[:,'MODEL_NAME']]

In [178]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
General                               3192
STD                                   1221
Childhood_Lead_Child_Model             361
Hep                                    296
ClusterModel                           195
Childhood_Lead_Investigation_Model     125
PortalApplicationModel                  30
Coinfection                             10
EHRAndScreeningModel_STD                 8
IsolationAndControlModel                 4
AggNETSS                                 2
Name: count, dtype: int64

In [179]:
#df_final[df_final['target_field'].isna()]

In [180]:
df_final.to_csv('output/maven_InterpretiveLab_origami_similarity_match.csv',index=False)